In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json
from sklearn import neighbors
from urllib.request import urlopen
%matplotlib inline

### Loading in dataframe that includes all data as well as addresses and GPS Coords.

In [6]:
df_full = pd.read_csv('data/allegations_precinct_address_included.csv')
df_full.head()

,unique_mos_id,first_name,last_name,command_now,shield_no,complaint_id,month_received,year_received,month_closed,year_closed,...,fado_type,allegation,precinct,contact_reason,outcome_description,board_disposition,borough,address,long,lat
0,10004,Jonathan,Ruiz,078 PCT,8409,42835,7,2019,5,2020,...,Abuse of Authority,Failure to provide RTKA card,78.0,Report-domestic dispute,No arrest made or summons issued,Substantiated (Command Lvl Instructions),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
1,10012,Paula,Smith,078 PCT,4021,37256,5,2017,10,2017,...,Abuse of Authority,Refusal to process civilian complaint,78.0,C/V telephoned PCT,No arrest made or summons issued,Substantiated (Command Lvl Instructions),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
2,10014,Malachy,Sullivan,078 PCT,4143,33969,11,2015,2,2016,...,Offensive Language,Sexual orientation,78.0,PD suspected C/V of violation/crime - street,Summons - other violation/crime,Substantiated (Formalized Training),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
3,10017,Fazle,Tanim,078 PCT,15187,40070,8,2018,11,2018,...,Discourtesy,Word,78.0,Moving violation,Moving violation summons issued,Unsubstantiated,Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
4,10017,Fazle,Tanim,078 PCT,15187,41927,3,2019,8,2019,...,Abuse of Authority,Refusal to provide shield number,78.0,Moving violation,Moving violation summons issued,Unsubstantiated,Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577


### Loading data layout table as a dataframe for reference 

In [7]:
dlo = pd.read_excel('data/CCRB Data Layout Table.xlsx')
dlo

,field name,description,glossary
0,unique_mos_id,"unique ID of the officer (""member of service"")",NaN
1,first_name,Officer's first name,NaN
2,last_name,Officer's last name,NaN
3,command_now,Officer's command assignment as of July 2020,See Tab 3
4,complaint_id,Unique ID of the complaint,NaN
5,month_received,Month the complaint was received by CCRB,NaN
6,year_received,Year the complaint was received by CCRB,NaN
7,month_closed,Month the complaint investigation was closed b...,NaN
8,year_closed,Year the complaint investigation was closed by...,NaN
9,command_at_incident,Officer's command assignment at the time of th...,NaN


In [8]:
df_full.head()

,unique_mos_id,first_name,last_name,command_now,shield_no,complaint_id,month_received,year_received,month_closed,year_closed,...,fado_type,allegation,precinct,contact_reason,outcome_description,board_disposition,borough,address,long,lat
0,10004,Jonathan,Ruiz,078 PCT,8409,42835,7,2019,5,2020,...,Abuse of Authority,Failure to provide RTKA card,78.0,Report-domestic dispute,No arrest made or summons issued,Substantiated (Command Lvl Instructions),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
1,10012,Paula,Smith,078 PCT,4021,37256,5,2017,10,2017,...,Abuse of Authority,Refusal to process civilian complaint,78.0,C/V telephoned PCT,No arrest made or summons issued,Substantiated (Command Lvl Instructions),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
2,10014,Malachy,Sullivan,078 PCT,4143,33969,11,2015,2,2016,...,Offensive Language,Sexual orientation,78.0,PD suspected C/V of violation/crime - street,Summons - other violation/crime,Substantiated (Formalized Training),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
3,10017,Fazle,Tanim,078 PCT,15187,40070,8,2018,11,2018,...,Discourtesy,Word,78.0,Moving violation,Moving violation summons issued,Unsubstantiated,Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
4,10017,Fazle,Tanim,078 PCT,15187,41927,3,2019,8,2019,...,Abuse of Authority,Refusal to provide shield number,78.0,Moving violation,Moving violation summons issued,Unsubstantiated,Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577


### Wrangle data to get df grouped by precincts, year, and fado type, making sure to include long/lat for mapping

In [60]:
## Group by precinct year, fado_type
precinct_grp_fado = df_full.groupby(by=['year_received', 'precinct', 'fado_type', 'address', 'long', 'lat', ],).size()

In [61]:
precinct_grp_fado.head()

year_received  precinct  fado_type           address                                      long        lat      
1985           63.0      Force               1844 Brooklyn Avenue, Brooklyn, NY, USA      -73.941616  40.627926    1
               67.0      Abuse of Authority  2820 Snyder Avenue, Brooklyn, NY, USA        -73.950264  40.648665    1
               71.0      Abuse of Authority  421 Empire Boulevard, Brooklyn, NY, USA      -73.947830  40.664592    1
                         Force               421 Empire Boulevard, Brooklyn, NY, USA      -73.947830  40.664592    1
               83.0      Abuse of Authority  480 Knickerbocker Avenue, Brooklyn, NY, USA  -73.917841  40.698147    1
dtype: int64

In [62]:
## Reset index to get a dataframe from broupby object 
precinct_grp_fado = precinct_grp_fado.reset_index()
precinct_grp_fado.head()

,year_received,precinct,fado_type,address,long,lat,0
0,1985,63.0,Force,"1844 Brooklyn Avenue, Brooklyn, NY, USA",-73.941616,40.627926,1
1,1985,67.0,Abuse of Authority,"2820 Snyder Avenue, Brooklyn, NY, USA",-73.950264,40.648665,1
2,1985,71.0,Abuse of Authority,"421 Empire Boulevard, Brooklyn, NY, USA",-73.947830,40.664592,1
3,1985,71.0,Force,"421 Empire Boulevard, Brooklyn, NY, USA",-73.947830,40.664592,1
4,1985,83.0,Abuse of Authority,"480 Knickerbocker Avenue, Brooklyn, NY, USA",-73.917841,40.698147,1


In [63]:
## Change name of size(count) column to #_complaints
precinct_grp_fado.rename(columns = {0:'#_complaints'}, inplace = True) 

In [64]:
# Filter to get dataframe only including 2002, for experiment with mapping 
precinct_grp_fado_2002 = precinct_grp_fado[precinct_grp_fado['year_received'] == 2002]

In [65]:
precinct_grp_fado_2002.shape

(147, 7)

In [66]:
precinct_grp_fado_2002.head()

,year_received,precinct,fado_type,address,long,lat,#_complaints
986,2002,1.0,Abuse of Authority,"16 Ericsson Place, Manhattan, NY, USA",-74.007198,40.720270,1
987,2002,1.0,Force,"16 Ericsson Place, Manhattan, NY, USA",-74.007198,40.720270,1
988,2002,5.0,Abuse of Authority,"19 Elizabeth Street, Manhattan, NY, USA",-73.997470,40.716194,1
989,2002,6.0,Abuse of Authority,"233 West 10 Street, Manhattan, NY, USA",-74.005453,40.734233,1
990,2002,6.0,Force,"233 West 10 Street, Manhattan, NY, USA",-74.005453,40.734233,1


In [67]:
# Sanity check 
precinct_grp_fado_2002.head()

,year_received,precinct,fado_type,address,long,lat,#_complaints
986,2002,1.0,Abuse of Authority,"16 Ericsson Place, Manhattan, NY, USA",-74.007198,40.720270,1
987,2002,1.0,Force,"16 Ericsson Place, Manhattan, NY, USA",-74.007198,40.720270,1
988,2002,5.0,Abuse of Authority,"19 Elizabeth Street, Manhattan, NY, USA",-73.997470,40.716194,1
989,2002,6.0,Abuse of Authority,"233 West 10 Street, Manhattan, NY, USA",-74.005453,40.734233,1
990,2002,6.0,Force,"233 West 10 Street, Manhattan, NY, USA",-74.005453,40.734233,1


### Use map box and plottly express to create bubble map to show: 
 - Count of complaints of given type associated with given precinct 
 - Use bubble size to show # Complaints 
 - Use buble color to show `fado_type`, meaning the type of complaint 

In [68]:
fig = px.scatter_geo(precinct_grp_fado_2002,
                    lat=precinct_grp_fado_2002.lat,
                    lon=precinct_grp_fado_2002.long,
                    locationmode='USA-states',
                    scope='usa',
                    center={'lat': 40.68230, 'lon': -73.98724},
                    hover_name="fado_type",
                    size='#_complaints',
                    color='fado_type'
                    )
fig.show()

### Improve map using Mapbox

In [69]:
## Access mapbox api using mapbox access token
with open("./mapbox_token", 'r') as f: 
    mapbox_key=f.read().strip()

In [70]:
## Create labels dictionary for hover info 
labels = {'fado_type': 'Complaint Type', 
         'precinct': 'Precinct', 
         'year_received': 'Complaint Year', 
         '#_complaints': 'No. Complaints'}
## Create list of columns that would appear upon hovering, using the labels above 
hover_data = {'precinct': True, 'year_received': True, 'fado_type': True, '#_complaints': True, 
             'long': False, 'lat': False}
## Create scatter mapbox using same dataframe 
fig = px.scatter_mapbox(precinct_grp_fado_2002, lat="lat", lon="long", 
                        size='#_complaints', color='fado_type',
                        labels=labels, hover_data=hover_data, opacity=0.6, zoom=9)
fig.update_layout(mapbox_style="light", mapbox_accesstoken=mapbox_key)
## Show Figure
fig.show()

### Create Animated map accross the years using precinct_grp_fado, without year filtering 

In [99]:
## Order dataframe by year 
precinct_grp_fado.sort_values(by=['year_received', 'precinct', 'fado_type'])
## Sanity check 
precinct_grp_fado.head()

,year_received,precinct,fado_type,address,long,lat,#_complaints
0,1985,63.0,Force,"1844 Brooklyn Avenue, Brooklyn, NY, USA",-73.941616,40.627926,1
1,1985,67.0,Abuse of Authority,"2820 Snyder Avenue, Brooklyn, NY, USA",-73.950264,40.648665,1
2,1985,71.0,Abuse of Authority,"421 Empire Boulevard, Brooklyn, NY, USA",-73.947830,40.664592,1
3,1985,71.0,Force,"421 Empire Boulevard, Brooklyn, NY, USA",-73.947830,40.664592,1
4,1985,83.0,Abuse of Authority,"480 Knickerbocker Avenue, Brooklyn, NY, USA",-73.917841,40.698147,1


In [102]:
## Create labels dictionary for hover info 
labels = {'fado_type': 'Complaint Type', 
         'precinct': 'Precinct', 
         'year_received': 'Complaint Year', 
         '#_complaints': 'No. Complaints'}
## Create list of columns that would appear upon hovering, using the labels above 
hover_data = {'precinct': True, 'year_received': True, 'fado_type': True, '#_complaints': True, 
             'long': False, 'lat': False}
## Create scatter mapbox using same dataframe 
fig = px.scatter_mapbox(precinct_grp_fado, lat="lat", lon="long", 
                        size='#_complaints', color='fado_type',
                        labels=labels, hover_data=hover_data, opacity=0.6,
                        animation_frame='year_received', animation_group='fado_type', zoom=9)
fig.update_layout(mapbox_style="light", mapbox_accesstoken=mapbox_key)
## Show Figure
fig.show()

#  <span style="color:red"> For some reason I can't get the animation above to include all 4 types of complaint typ. </span>

### Animation across years for number of complaints only 
 - color based on boroughs

In [75]:
## Reference df 
df_full.head()

,unique_mos_id,first_name,last_name,command_now,shield_no,complaint_id,month_received,year_received,month_closed,year_closed,...,fado_type,allegation,precinct,contact_reason,outcome_description,board_disposition,borough,address,long,lat
0,10004,Jonathan,Ruiz,078 PCT,8409,42835,7,2019,5,2020,...,Abuse of Authority,Failure to provide RTKA card,78.0,Report-domestic dispute,No arrest made or summons issued,Substantiated (Command Lvl Instructions),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
1,10012,Paula,Smith,078 PCT,4021,37256,5,2017,10,2017,...,Abuse of Authority,Refusal to process civilian complaint,78.0,C/V telephoned PCT,No arrest made or summons issued,Substantiated (Command Lvl Instructions),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
2,10014,Malachy,Sullivan,078 PCT,4143,33969,11,2015,2,2016,...,Offensive Language,Sexual orientation,78.0,PD suspected C/V of violation/crime - street,Summons - other violation/crime,Substantiated (Formalized Training),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
3,10017,Fazle,Tanim,078 PCT,15187,40070,8,2018,11,2018,...,Discourtesy,Word,78.0,Moving violation,Moving violation summons issued,Unsubstantiated,Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
4,10017,Fazle,Tanim,078 PCT,15187,41927,3,2019,8,2019,...,Abuse of Authority,Refusal to provide shield number,78.0,Moving violation,Moving violation summons issued,Unsubstantiated,Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577


In [78]:
## Group by year, borough, precinct
precinct_grp = df_full.groupby(by=['year_received', 'borough', 'precinct', 'address', 'long', 'lat', ],).size()

In [79]:
## Sanity check 
precinct_grp.head()


year_received  borough   precinct  address                                      long        lat      
1985           Brooklyn  63.0      1844 Brooklyn Avenue, Brooklyn, NY, USA      -73.941616  40.627926    1
                         67.0      2820 Snyder Avenue, Brooklyn, NY, USA        -73.950264  40.648665    1
                         71.0      421 Empire Boulevard, Brooklyn, NY, USA      -73.947830  40.664592    2
                         83.0      480 Knickerbocker Avenue, Brooklyn, NY, USA  -73.917841  40.698147    3
1986           Brooklyn  67.0      2820 Snyder Avenue, Brooklyn, NY, USA        -73.950264  40.648665    4
dtype: int64

In [80]:
## Get dataframe for groupby object 
precinct_grp = precinct_grp.reset_index()
precinct_grp.head()

,year_received,borough,precinct,address,long,lat,0
0,1985,Brooklyn,63.0,"1844 Brooklyn Avenue, Brooklyn, NY, USA",-73.941616,40.627926,1
1,1985,Brooklyn,67.0,"2820 Snyder Avenue, Brooklyn, NY, USA",-73.950264,40.648665,1
2,1985,Brooklyn,71.0,"421 Empire Boulevard, Brooklyn, NY, USA",-73.947830,40.664592,2
3,1985,Brooklyn,83.0,"480 Knickerbocker Avenue, Brooklyn, NY, USA",-73.917841,40.698147,3
4,1986,Brooklyn,67.0,"2820 Snyder Avenue, Brooklyn, NY, USA",-73.950264,40.648665,4


In [81]:
## Change name of size(count) column to #_complaints
precinct_grp.rename(columns = {0:'#_complaints'}, inplace = True) 

In [82]:
## Sanity check 
precinct_grp.head()

,year_received,borough,precinct,address,long,lat,#_complaints
0,1985,Brooklyn,63.0,"1844 Brooklyn Avenue, Brooklyn, NY, USA",-73.941616,40.627926,1
1,1985,Brooklyn,67.0,"2820 Snyder Avenue, Brooklyn, NY, USA",-73.950264,40.648665,1
2,1985,Brooklyn,71.0,"421 Empire Boulevard, Brooklyn, NY, USA",-73.947830,40.664592,2
3,1985,Brooklyn,83.0,"480 Knickerbocker Avenue, Brooklyn, NY, USA",-73.917841,40.698147,3
4,1986,Brooklyn,67.0,"2820 Snyder Avenue, Brooklyn, NY, USA",-73.950264,40.648665,4


In [84]:
## Order dataframe by year 
precinct_grp.sort_values('year_received', inplace=True)

In [108]:
precinct_grp['borough'].value_counts()

Brooklyn          579
Manhattan         489
Queens            329
Bronx             304
Staten Island      41
Staten Island      29
Name: borough, dtype: int64

In [112]:
## Create labels dictionary for hover info 
labels = {'precinct': 'Precinct', 
         'year_received': 'Complaint Year', 
         '#_complaints': 'No. Complaints', 
         'borough': 'Borough'}
## Create list of columns that would appear upon hovering, using the labels above 
hover_data = {'precinct': True, 'year_received': True, 'borough': True, '#_complaints': True, 
             'long': False, 'lat': False}

## Create scatter mapbox using same dataframe 
fig = px.scatter_mapbox(precinct_grp, lat="lat", lon="long", 
                        size='#_complaints',
                        labels=labels, hover_data=hover_data, opacity=0.6,
                        animation_frame='year_received', zoom=8,)
fig.update_layout(mapbox_style="light", mapbox_accesstoken=mapbox_key)
## Show Figure
fig.show()

#  <span style="color:red"> When I add `color='borough'` it only shows Brooklyn. Don't know why, but very similar issue to the one mentioned above </span>